# Amazon ML Challenge 2026 — Clean Entity Resolution Pipeline

Restartable notebook for the current pipeline:
**V2 normalization → V2 threshold-1000/top-k-2 blocking → matcher features → LightGBM → validation → save model.**

Older V1, threshold-500 and threshold-750 experiments are intentionally excluded.

## 0. Imports and configuration

In [1]:
from pathlib import Path
import re, unicodedata, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, fbeta_score
from lightgbm import LGBMClassifier
from rapidfuzz.fuzz import ratio, WRatio


RANDOM_STATE = 42


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path

    raise RuntimeError(
        f"Could not find project root from {start}"
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "output"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory :", Path.cwd())
print("Project root              :", PROJECT_ROOT)
print("Data directory            :", DATA_DIR)
print("Raw data                  :", RAW_DIR)
print("Interim data              :", INTERIM_DIR)
print("Processed data            :", PROCESSED_DIR)
print("Model directory           :", MODEL_DIR)
print("Output directory          :", OUTPUT_DIR)

Current working directory : d:\amzn_ml_challenge\notebooks
Project root              : D:\amzn_ml_challenge
Data directory            : D:\amzn_ml_challenge\data
Raw data                  : D:\amzn_ml_challenge\data\raw
Interim data              : D:\amzn_ml_challenge\data\interim
Processed data            : D:\amzn_ml_challenge\data\processed
Model directory           : D:\amzn_ml_challenge\models
Output directory          : D:\amzn_ml_challenge\output


## 1. Load training data

In [2]:
s1_train = pd.read_csv(RAW_DIR / "train_source1.tsv", sep="\t", dtype=str)
s2_train = pd.read_csv(RAW_DIR / "train_source2.tsv", sep="\t", dtype=str)
s3_train = pd.read_csv(RAW_DIR / "train_source3.tsv", sep="\t", dtype=str)
ground_truth = pd.read_csv(RAW_DIR / "train_ground_truth.tsv", sep="\t", dtype=str)

print(s1_train.shape, s2_train.shape, s3_train.shape, ground_truth.shape)

(2206821, 4) (5034616, 4) (5285603, 4) (2206821, 2)


## 2. Ground-truth parsing and stratified validation split

In [3]:
def parse_gt_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(",") if x.strip()]

ground_truth["match_count"] = ground_truth["matched_entity_ids"].map(
    lambda x: len(parse_gt_ids(x))
)

train_ids, valid_ids = train_test_split(
    ground_truth["source1_entity_id"],
    test_size=0.20,
    stratify=ground_truth["match_count"],
    random_state=RANDOM_STATE,
)

train_s1_ids = set(train_ids)
valid_s1_ids = set(valid_ids)
valid_gt = ground_truth[ground_truth["source1_entity_id"].isin(valid_s1_ids)].copy()
valid_gt_lookup = dict(zip(valid_gt["source1_entity_id"], valid_gt["matched_entity_ids"]))

print("Train S1:", len(train_s1_ids))
print("Valid S1:", len(valid_s1_ids))

Train S1: 1765456
Valid S1: 441365


## 3. V2 normalization

In [5]:
def normalize_name_v2(value):
    if pd.isna(value): return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.casefold()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

def normalize_address_v2(value):
    if pd.isna(value): return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.casefold()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    replacements = {
        r"\bave\b":"avenue", r"\bav\b":"avenue", r"\bst\b":"street",
        r"\brd\b":"road", r"\bdr\b":"drive", r"\bln\b":"lane",
        r"\btrl\b":"trail", r"\bhwy\b":"highway", r"\bblvd\b":"boulevard",
        r"\bpkwy\b":"parkway", r"\bct\b":"court", r"\bpl\b":"place",
        r"\bcir\b":"circle", r"\bter\b":"terrace", r"\bste\b":"suite",
        r"\bapt\b":"apartment",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
    return re.sub(r"\s+", " ", text).strip()

for df in (s1_train, s2_train, s3_train):
    df["normalized_name_v2"] = df["business_name"].map(normalize_name_v2)
    df["normalized_address_v2"] = df["business_address"].map(normalize_address_v2)

validation_s1 = s1_train[
    s1_train["entity_id"].isin(valid_s1_ids)
].copy()

print("V2 normalization complete:", validation_s1.shape)

V2 normalization complete: (441365, 6)


## 4. Build blocking indexes

Locked configuration: token frequency threshold = **1000**, top-k rare tokens = **2**.

In [6]:
TOKEN_FREQ_THRESHOLD = 1000
TOP_K = 2

def build_token_index(df, column, max_frequency):
    counts = {}
    for value in df[column]:
        for token in set(str(value).split()) if value else set():
            counts[token] = counts.get(token, 0) + 1

    index = {}
    for entity_id, value in zip(df["entity_id"], df[column]):
        for token in set(str(value).split()) if value else set():
            if counts[token] <= max_frequency:
                index.setdefault(token, []).append(entity_id)
    return index, counts

s2_name_index, s2_name_freq = build_token_index(s2_train, "normalized_name_v2", TOKEN_FREQ_THRESHOLD)
s2_address_index, s2_address_freq = build_token_index(s2_train, "normalized_address_v2", TOKEN_FREQ_THRESHOLD)
s3_name_index, s3_name_freq = build_token_index(s3_train, "normalized_name_v2", TOKEN_FREQ_THRESHOLD)
s3_address_index, s3_address_freq = build_token_index(s3_train, "normalized_address_v2", TOKEN_FREQ_THRESHOLD)

s2_name_exact = s2_train.groupby("normalized_name_v2")["entity_id"].apply(list).to_dict()
s2_address_exact = s2_train.groupby("normalized_address_v2")["entity_id"].apply(list).to_dict()
s3_name_exact = s3_train.groupby("normalized_name_v2")["entity_id"].apply(list).to_dict()
s3_address_exact = s3_train.groupby("normalized_address_v2")["entity_id"].apply(list).to_dict()

print("Indexes ready.")

Indexes ready.


## 5. Locked blocker

In [7]:
def generate_v2_block_candidates_1000(s1_row, top_k=TOP_K):
    name = s1_row["normalized_name_v2"]
    address = s1_row["normalized_address_v2"]

    s2, s3 = set(), set()

    if name:
        s2.update(s2_name_exact.get(name, []))
        s3.update(s3_name_exact.get(name, []))
    if address:
        s2.update(s2_address_exact.get(address, []))
        s3.update(s3_address_exact.get(address, []))

    name_tokens = set(name.split()) if name else set()
    address_tokens = set(address.split()) if address else set()

    rare_name = sorted(
        [t for t in name_tokens if s2_name_freq.get(t, 10**9) <= TOKEN_FREQ_THRESHOLD],
        key=lambda t: s2_name_freq.get(t, 10**9)
    )[:top_k]

    rare_address = sorted(
        [t for t in address_tokens if s2_address_freq.get(t, 10**9) <= TOKEN_FREQ_THRESHOLD],
        key=lambda t: s2_address_freq.get(t, 10**9)
    )[:top_k]

    for token in rare_name:
        s2.update(s2_name_index.get(token, []))
        s3.update(s3_name_index.get(token, []))
    for token in rare_address:
        s2.update(s2_address_index.get(token, []))
        s3.update(s3_address_index.get(token, []))

    return list(s2) + list(s3)

## 6. Validate blocking on the full validation set

In [9]:
candidate_counts = []

recovered = total = zero = 0

for row in validation_s1.itertuples(index=False):

    candidates = set(generate_v2_block_candidates_1000({
        "normalized_name_v2": row.normalized_name_v2,
        "normalized_address_v2": row.normalized_address_v2,
    }))

    gt = set(parse_gt_ids(valid_gt_lookup[row.entity_id]))

    total += len(gt)
    recovered += len(gt & candidates)
    candidate_counts.append(len(candidates))
    zero += not candidates

blocking_metrics = {
    "pair_recall": recovered / total,
    "recovered_pairs": recovered,
    "total_gt_pairs": total,
    "mean_candidates": np.mean(candidate_counts),
    "median_candidates": np.median(candidate_counts),
    "p95_candidates": np.percentile(candidate_counts, 95),
    "zero_candidate_s1": int(zero),
}

print(pd.Series(blocking_metrics))

pair_recall          9.070472e-01
recovered_pairs      1.385677e+06
total_gt_pairs       1.527679e+06
mean_candidates      1.121278e+03
median_candidates    8.810000e+02
p95_candidates       3.068000e+03
zero_candidate_s1    2.440000e+03
dtype: float64


## 7. Build 10k-S1 matcher experiment

In [11]:
matcher_sample = validation_s1.sample(
    n=min(10_000, len(validation_s1)),
    random_state=RANDOM_STATE
).copy()

candidate_rows = []
for row in matcher_sample.itertuples(index=False):
    for target_id in generate_v2_block_candidates_1000({
        "normalized_name_v2": row.normalized_name_v2,
        "normalized_address_v2": row.normalized_address_v2,
    }):
        candidate_rows.append((row.entity_id, target_id))

candidate_pairs_df = pd.DataFrame(
    candidate_rows,
    columns=["source1_entity_id", "target_entity_id"]
)
print("Candidate pairs:", len(candidate_pairs_df))

Candidate pairs: 11134489


## 8. Join source fields

In [14]:
s1f = matcher_sample[[
    "entity_id",
    "business_name",
    "business_address",
    "country",
    "normalized_name_v2",
    "normalized_address_v2"
]].rename(columns={
    "entity_id": "source1_entity_id",
    "business_name": "s1_business_name",
    "business_address": "s1_business_address",
    "country": "s1_country",
    "normalized_name_v2": "s1_name_v2",
    "normalized_address_v2": "s1_address_v2"
})


def target_frame(df):
    return df[[
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "normalized_name_v2",
        "normalized_address_v2"
    ]].rename(columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
        "normalized_name_v2": "target_name_v2",
        "normalized_address_v2": "target_address_v2"
    })


s2f = target_frame(s2_train)
s3f = target_frame(s3_train)


candidate_features = candidate_pairs_df.merge(
    s1f,
    on="source1_entity_id",
    how="left",
    validate="many_to_one"
)

candidate_features["target_source"] = (
    candidate_features["target_entity_id"].str[:2]
)

c2 = candidate_features[
    candidate_features["target_source"] == "S2"
].drop(columns="target_source")

c3 = candidate_features[
    candidate_features["target_source"] == "S3"
].drop(columns="target_source")


c2 = c2.merge(
    s2f,
    on="target_entity_id",
    how="left",
    validate="many_to_one"
)

c3 = c3.merge(
    s3f,
    on="target_entity_id",
    how="left",
    validate="many_to_one"
)


candidate_features = pd.concat(
    [c2, c3],
    ignore_index=True
)

print("candidate_features:", candidate_features.shape)
print(candidate_features.columns.tolist())

candidate_features: (11134489, 12)
['source1_entity_id', 'target_entity_id', 's1_business_name', 's1_business_address', 's1_country', 's1_name_v2', 's1_address_v2', 'target_business_name', 'target_business_address', 'target_country', 'target_name_v2', 'target_address_v2']


## 9. Matching features

In [15]:
candidate_features["country_same"] = (
    candidate_features["s1_country"] == candidate_features["target_country"]
).astype("int8")
candidate_features["name_exact"] = (
    candidate_features["s1_name_v2"] == candidate_features["target_name_v2"]
).astype("int8")
candidate_features["address_exact"] = (
    candidate_features["s1_address_v2"] == candidate_features["target_address_v2"]
).astype("int8")

candidate_features["name_length_diff"] = (
    candidate_features["s1_name_v2"].str.len() -
    candidate_features["target_name_v2"].str.len()
).abs()
candidate_features["address_length_diff"] = (
    candidate_features["s1_address_v2"].str.len() -
    candidate_features["target_address_v2"].str.len()
).abs()

def token_jaccard(a,b):
    if not a or not b: return 0.0
    A,B=set(a.split()),set(b.split())
    U=A|B
    return len(A&B)/len(U) if U else 0.0

def token_containment(a,b):
    if not a or not b: return 0.0
    A,B=set(a.split()),set(b.split())
    return len(A&B)/min(len(A),len(B)) if A and B else 0.0

for prefix, a_col, b_col in [
    ("name","s1_name_v2","target_name_v2"),
    ("address","s1_address_v2","target_address_v2")
]:
    candidate_features[f"{prefix}_token_jaccard"] = [
        token_jaccard(a,b) for a,b in zip(candidate_features[a_col],candidate_features[b_col])
    ]
    candidate_features[f"{prefix}_token_containment"] = [
        token_containment(a,b) for a,b in zip(candidate_features[a_col],candidate_features[b_col])
    ]

candidate_features["name_char_ratio"] = [
    ratio(a,b)/100 for a,b in zip(candidate_features["s1_name_v2"],candidate_features["target_name_v2"])
]
candidate_features["address_char_ratio"] = [
    ratio(a,b)/100 for a,b in zip(candidate_features["s1_address_v2"],candidate_features["target_address_v2"])
]
candidate_features["name_wratio"] = [
    WRatio(a,b)/100 for a,b in zip(candidate_features["s1_name_v2"],candidate_features["target_name_v2"])
]
candidate_features["address_wratio"] = [
    WRatio(a,b)/100 for a,b in zip(candidate_features["s1_address_v2"],candidate_features["target_address_v2"])
]

print("Features complete.")

Features complete.


## 10. Label candidate pairs

In [17]:
positive_pairs = set()

for s1_id in set(matcher_sample["entity_id"]):

    for target_id in parse_gt_ids(valid_gt_lookup[s1_id]):
        positive_pairs.add((s1_id, target_id))


positive_df = pd.DataFrame(
    list(positive_pairs),
    columns=["source1_entity_id", "target_entity_id"]
).assign(is_match=1)


candidate_features = candidate_features.merge(
    positive_df,
    on=["source1_entity_id", "target_entity_id"],
    how="left"
)

candidate_features["is_match"] = (
    candidate_features["is_match"]
    .fillna(0)
    .astype("int8")
)

print(candidate_features["is_match"].value_counts())

is_match
0    11103251
1       31238
Name: count, dtype: int64


## 11. Train LightGBM matcher

In [18]:
feature_cols = [
    "country_same","name_exact","address_exact",
    "name_length_diff","address_length_diff",
    "name_token_jaccard","name_token_containment",
    "address_token_jaccard","address_token_containment",
    "name_char_ratio","address_char_ratio",
    "name_wratio","address_wratio",
]

model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(candidate_features[feature_cols], candidate_features["is_match"])
candidate_features["match_probability"] = model.predict_proba(
    candidate_features[feature_cols]
)[:,1]
print("Model trained.")

[LightGBM] [Info] Number of positive: 31238, number of negative: 11103251
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060269 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1562
[LightGBM] [Info] Number of data points in the train set: 11134489, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Model trained.


## 12. Pair-level threshold evaluation

In [19]:
thresholds = [0.990,0.992,0.994,0.996,0.998,0.999]
y_true = candidate_features["is_match"].values
probs = candidate_features["match_probability"].values

rows=[]
for t in thresholds:
    pred=(probs>=t).astype("int8")
    rows.append({
        "threshold":t,
        "predicted_matches":int(pred.sum()),
        "precision":precision_score(y_true,pred,zero_division=0),
        "recall":recall_score(y_true,pred,zero_division=0),
        "F0.5":fbeta_score(y_true,pred,beta=0.5,zero_division=0),
    })
pair_results=pd.DataFrame(rows)
print(pair_results)

   threshold  predicted_matches  precision    recall      F0.5
0      0.990              35926   0.803875  0.924515  0.825416
1      0.992              34621   0.823084  0.912222  0.839490
2      0.994              33010   0.846683  0.894712  0.855872
3      0.996              30751   0.877565  0.863884  0.874794
4      0.998              26805   0.926282  0.794833  0.896626
5      0.999              21512   0.972155  0.669473  0.891539


## 13. Set-level evaluation on the 10k matcher sample

In [21]:
sample_s1_ids = set(matcher_sample["entity_id"])

rows = []

for t in thresholds:

    pred_sets = (
        candidate_features[
            candidate_features["match_probability"] >= t
        ]
        .groupby("source1_entity_id")["target_entity_id"]
        .apply(set)
    )

    tp = fp = fn = 0

    for s1_id in sample_s1_ids:

        gt = set(parse_gt_ids(valid_gt_lookup[s1_id]))
        pred = pred_sets.get(s1_id, set())

        tp += len(pred & gt)
        fp += len(pred - gt)
        fn += len(gt - pred)

    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0

    f05 = (
        1.25 * p * r / (0.25 * p + r)
        if (p + r)
        else 0
    )

    rows.append({
        "threshold": t,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": p,
        "recall": r,
        "F0.5": f05,
    })

set_results = pd.DataFrame(rows)

print(set_results)

   threshold     TP    FP     FN  precision    recall      F0.5
0      0.990  28880  7046   5596   0.803875  0.837684  0.810416
1      0.992  28496  6125   5980   0.823084  0.826546  0.823774
2      0.994  27949  5061   6527   0.846683  0.810680  0.839229
3      0.996  26986  3765   7490   0.877565  0.782747  0.856807
4      0.998  24829  1976   9647   0.926282  0.720182  0.876136
5      0.999  20913   599  13563   0.972155  0.606596  0.867587


## 14. Save model and configuration

In [22]:
config={
    "random_state":RANDOM_STATE,
    "blocking":{"normalization":"V2","token_frequency_threshold":1000,"top_k":2},
    "matcher_features":feature_cols,
    "recommended_threshold_from_current_10k_experiment":0.998,
}

with open(MODEL_DIR/"entity_matcher_lightgbm.pkl","wb") as f:
    pickle.dump(model,f)
with open(MODEL_DIR/"matcher_config.pkl","wb") as f:
    pickle.dump(config,f)

print("Saved model and configuration to", MODEL_DIR)

Saved model and configuration to D:\amzn_ml_challenge\models


## 15. Next phase

The notebook now ends after the validated matcher experiment.

For the final submission we still need a **separate production inference notebook/script** that:
- loads the saved matcher,
- processes all test S1 rows,
- generates test candidates with the locked blocker,
- scores them,
- applies the selected threshold,
- writes `matching_results.tsv`,
- validates it with the official validator.

Keeping inference separate prevents the experimental notebook from becoming another huge, fragile notebook.